# Using `data-sampler` as a Python package

This notebook walks through the library API on the bundled dummy dataset
(`examples/employees.csv`): load a file, inspect column stats, draw a
representative (stratified) sample, anonymize chosen columns, and save.

Install (not on PyPI yet — use the GitHub release):

```sh
pip install https://github.com/aaronified/data-sampler/releases/download/v3.0.1/data_sampler-3.0.1-py3-none-any.whl
```

In [1]:
import os

os.environ.setdefault("DATA_SAMPLER_LOG", "quiet")  # keep log lines out of cell outputs

import data_sampler as ds

print("data-sampler", ds.__version__)

data-sampler 3.0.1


## 1. Load a data file

`load_file` handles CSV, TSV, JSON, Excel (`sheet=` picks the sheet), and Parquet.

In [2]:
from pathlib import Path

src = Path("employees.csv")
if not src.exists():
    src = Path("examples/employees.csv")  # when running from the repo root

df = ds.load_file(src)
print(f"{len(df):,} rows \u00d7 {len(df.columns)} columns")
df.head()

1,000 rows × 8 columns


,employee_id,full_name,email,department,region,employment_type,performance_rating,salary
0,E1001,Emily Lee,emily.lee001@example.com,Sales,North,Full-time,4,62000
1,E1002,Joshua Clark,joshua.clark002@example.com,Finance,South,Full-time,4,50000
2,E1003,Donald Martin,donald.martin003@example.com,Operations,East,Contract,3,68500
3,E1004,Anthony Garcia,anthony.garcia004@example.com,Sales,East,Full-time,4,64000
4,E1005,Donna Johnson,donna.johnson005@example.com,Engineering,East,Full-time,3,74000


## 2. Column statistics

`compute_stats` returns one `ColumnStats` per column — the same numbers the
TUI shows. `stratifiable` tells you which columns the auto-stratifier would
consider when preserving statistical variety.

In [3]:
import pandas as pd

pd.DataFrame(
    {
        "column": s.name,
        "kind": s.kind,
        "missing %": round(s.missing_pct, 1),
        "unique": s.unique,
        "distribution": ds.sparkline(s.histogram),
        "summary": s.summary(),
        "stratifiable": s.stratifiable,
    }
    for s in ds.compute_stats(df)
)

,column,kind,missing %,unique,distribution,summary,stratifiable
0,employee_id,categorical,0.0,1000,████████,top: E1001 (0%),False
1,full_name,categorical,0.0,755,█▆▆▆▆▆▅▅,top: Andrew Jones (0%),False
2,email,categorical,0.0,1000,████████,top: emily.lee001@example (0%),False
3,department,categorical,0.0,5,█▅▄▂▂,top: Engineering (36%),True
4,region,categorical,0.0,4,█▆▄▂,top: North (38%),True
5,employment_type,categorical,0.0,3,█▃▂,top: Full-time (69%),True
6,performance_rating,numeric,0.0,5,▂▁▃▁▁█▁▆▁▂,1 … 5 μ=3.23,True
7,salary,numeric,0.0,189,▂▇█▄▂▂▂▂▁▂,"28,000 … 234,000 μ=76,618",False


## 3. Draw a representative sample

`sample` stratifies automatically on suitable columns. Use
`exclude_columns=[...]` to keep specific columns out of stratification, and
`random_state` for reproducibility. The returned `SampleResult` carries the
sampled frame plus everything needed for the report.

In [4]:
result = ds.sample(df, 100, exclude_columns=["performance_rating"], random_state=42)

print("method:", result.method)
print("stratified on:", result.strat_cols)
print(ds.format_stratification_report(df, result))

method: stratified
stratified on: ['employment_type', 'region', 'department']
╔═════════════════════════════════════════════════════════════════════╗
║                        STRATIFICATION REPORT                        ║
╚═════════════════════════════════════════════════════════════════════╝

  Columns used: employment_type, region, department

  Method: rows are grouped by the intersection of all stratification
  columns simultaneously (e.g. A=x AND B=y AND C=z). Proportional
  allocation is then computed per intersection group. A category value
  that is small in every intersection group it appears in will receive
  0 allocation — even if it looks sizable in the per-column view below.

  Column: 'employment_type' (3 categories)
          Value          Original                    Sample          
  ─────────────────────────────────────────────────────────────────────
       Contract  ██░░░░░░░░░░░░░  10.1%  █░░░░░░░░░░░░░░   9.0%
      Full-time  ███████████████  68.9%  ████████████

The sample tracks the original distribution closely — compare proportions directly:

In [5]:
pd.DataFrame(
    {
        "original %": (df["department"].value_counts(normalize=True) * 100).round(1),
        "sample %": (result.data["department"].value_counts(normalize=True) * 100).round(1),
    }
)

,original %,sample %
department,,
Engineering,36.5,38.0
Sales,24.8,26.0
Operations,18.8,19.0
Finance,10.0,9.0
HR,9.9,8.0


## 4. Anonymize chosen columns

Each anonymizer maps every *unique* original value to exactly one
replacement, so repeated values stay repeated and distributions survive.
Missing values pass through. Specs can be a kind string, a
`(kind, options)` tuple, a dict, or an anonymizer instance.

In [6]:
anon = ds.anonymize(
    result.data,
    {
        "full_name": "names",                              # bundled name library
        "employee_id": ("sequential_id", {"start": 1000}),  # 1000, 1001, ...
        "salary": "numeric_jitter",                        # \u00b120% by default
        "email": {"kind": "hex", "length": 10},            # hex strings
    },
    seed=42,
)
anon[["employee_id", "full_name", "email", "department", "salary"]].head()

,employee_id,full_name,email,department,salary
744,1000,Ravi Andersen,6a78c49ea2,Engineering,62264
371,1001,Thomas Gomez,0e32684b27,Engineering,102743
308,1002,Fatima Singh,b95e909348,Operations,46793
346,1003,Benjamin Smith,334896a68f,Operations,55882
894,1004,Nia Perez,812d810a48,Operations,44554


In [7]:
# the statistical guarantees, verified:
# 1. one replacement per unique original (repeats stay repeats)
replacements_per_original = (
    pd.DataFrame({"orig": result.data["full_name"], "anon": anon["full_name"]})
    .groupby("orig")["anon"].nunique()
)
assert (replacements_per_original == 1).all()

# 2. jittered salaries stay within \u00b120% of the originals
ratio = anon["salary"] / result.data["salary"]
assert ratio.between(0.8, 1.2).all()

print("verified: consistent mapping + jitter within \u00b120%")

verified: consistent mapping + jitter within ±20%


## 5. Save the result

`save_output` keeps the source format and names the file
`{stem}_{tag}{ext}`. Omit `output_folder` to save next to the source file
(here we use a temp dir to keep the repo clean).

In [8]:
import tempfile

out_path = ds.save_output(anon, src, tag="sample_100_anon", output_folder=tempfile.gettempdir())
print("saved as:", out_path.name, "(in the system temp folder)")

saved as: employees_sample_100_anon.csv (in the system temp folder)


## Prefer the visual route?

The same workflow is available in the terminal UI:

- `data-sampler` (no arguments) or `ds.run_tui()` — note: the TUI needs a
  real terminal, so run it from a shell rather than from this notebook.
- Launcher scripts: `scripts/run-tui.sh` (Linux/macOS) and
  `scripts/run-tui.bat` (Windows).